# MCP Server + Client — Sample Code (template)
### Module 7 sample notebook

The smallest end-to-end MCP example: one tool on a **server**, and a **client** that launches it, lists the tool, and calls it. Copy this as the skeleton for Mini-Project 5. See `mcp_part1.ipynb` / `mcp_part2.ipynb` for the full lessons. `pip install mcp`.

**Step 1 — write a tiny server to `tiny_server.py`** (one tool):

In [ ]:
server_src = """
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("tiny", log_level="WARNING")

@mcp.tool()
def whois(host: str) -> str:
    "Return a (fake) owner for a host."
    return f"{host} is owned by ExampleCorp (simulated)."

if __name__ == "__main__":
    mcp.run()   # stdio transport
"""
open("tiny_server.py", "w", encoding="utf-8").write(server_src)
print("wrote tiny_server.py")

**Step 2 — a client launches it, lists the tool, and calls it** (Jupyter-safe async):

In [ ]:
import sys, asyncio, threading
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def run_async(coro):
    box = {}
    def worker():
        loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
        try: box["v"] = loop.run_until_complete(coro)
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    return box["v"]

server = StdioServerParameters(command=sys.executable, args=["tiny_server.py"])

async def demo():
    errlog = open("tiny_server_stderr.log", "w")     # real file (avoids Jupyter fileno error)
    async with stdio_client(server, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("tools:", [t.name for t in tools.tools])
            out = await session.call_tool("whois", {"host": "10.0.0.5"})
            print("result:", out.content[0].text)

run_async(demo())